# Q2: Vision Language Models (VLMs)

# S1: VLM Anatomy

**Objective:** In this section, you will learn the anatomy of Vision-Language Models (VLMs), identify their core components, and practically explore how images are translated into tokens that a Large Language Model (LLM) can process.

## 1. Model Anatomy

A typical Vision-Language Model usually consists of three main components:

- **Vision Encoder:** A neural network (often a Vision Transformer or ViT) that processes the raw image to extract visual features.
- **Vision-Language Projector:** Linear layers or MLPs that map the extracted visual features into the same dimensional space as the text tokens.
- **Large Language Model:** The core LLM that processes the combined sequence of text and visual tokens to generate an output.



### Task 1.1: Load and Inspect the Model



### Import Libraries

In [ ]:
import time
import random
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import (
    AutoProcessor, 
    AutoModelForCausalLM, 
    TrainingArguments, 
    Trainer
)
from peft import LoraConfig, get_peft_model
from qwen_vl_utils import process_vision_info

Complete and run the code cell below to use the transformers library to load the Qwen/Qwen3.5-0.8B model and its processor, and print the underlying model architecture.


In [ ]:
model_id = "Qwen/Qwen3.5-VL-0.8B"

# TODO: Load processor
processor = ...

# TODO: Load model (use bfloat16 or float16 to save memory)
model = ...

print(model)

**Question 1.1:** Inspect the printed model architecture layout above. Identify and write down the exact module or class names corresponding to the Vision Encoder, the Projector, and the Large Language Model components.

**Student's Answer:**


## 2. Baseline Inference

Let's test the model on real data to see how it performs. We will break this down into three steps: loading data, formatting the multimodal prompts, and generating the responses.

### Important Note: Non-Thinking Mode

For all evaluations and fine-tuning tasks in this notebook, you must set the model to **Non-thinking mode** to test its pure Instruct behavior. 

For technical details, check the [Hugging Face Qwen3.5 Documentation](https://huggingface.co/docs/transformers/main/en/model_doc/qwen2_5_vl).

### Task 1.2a: Load and Explore the Dataset

Complete and run the code below to load the ScienceQA dataset, filter it for samples containing image, and select 3 samples of your choice.

In [ ]:
dataset = load_dataset("derek-thomas/ScienceQA")

# TODO: Filter the 'train' split to keep ONLY samples with an 'image'
image_data = ...

# TODO: Select 3 samples of your choice from image_data
selected_samples = ...
print(selected_samples)

### Task 1.2b: Format the Multimodal Prompts

VLMs require structural preprocessing to combine image arrays and text strings into a unified sequence. Complete and run the cell below to apply the chat template and process the multimodal sequence for the selected evaluation samples.


In [ ]:
prepared_samples = []

for sample in selected_samples:
    question = sample['question']
    choices = sample['choices']
    answer_idx = sample['answer']

    ground_truth = str(choices[answer_idx]).strip()
    choices_text = "\n".join([f"- {c}" for c in choices])
    full_prompt = f"{question}\nChoices:\n{choices_text}\nAnswer concisely with the correct choice."

    # TODO: Build the messages list (user role, containing image and text)
    messages = ...

    # TODO: Apply chat template 
    text = ...

    # TODO: Extract image inputs using process_vision_info
    image_inputs, _ = ...

    # TODO: Get final inputs using processor
    inputs = ...

    prepared_samples.append({
        "inputs": inputs,
        "image": sample["image"],
        "prompt": full_prompt,
        "ground_truth": ground_truth
    })

print(f"Successfully formatted {len(prepared_samples)} prompts.")

### Task 1.2c: Generate and Decode

In this task, you will execute the baseline generation. Complete and run the loop below to feed the preprocessed inputs into the model, extract the newly generated IDs, and decode them into readable text answers.

**Note:** You must write your code to explicitly display the Image, the Text Prompt/Question, the Ground Truth Answer, and the generated Model Output sequentially for each sample. This is a mandatory step to directly assess and verify the VLM's inference capabilities.


In [ ]:
for sample_data in prepared_samples:
    # Safely move inputs to the same device as the model
    inputs = sample_data["inputs"].to(model.device)
    
    # TODO: Generate output IDs using the model (inside torch.no_grad())
    ...

    # TODO: Decode ONLY the newly generated tokens to text
    output_text = ...

    # Display results
    plt.figure(figsize=(4, 4))
    plt.imshow(sample_data["image"])
    plt.axis("off")
    plt.show()
    print(f"Prompt: {sample_data['prompt']}\nGround Truth: {sample_data['ground_truth']}\nModel Output: {output_text}\n" + "-"*50)

## 3. Playing with Tokens: Analyzing the Input Sequence

When a VLM encounters an image, it converts the picture into a series of 'visual tokens' and integrates them with the text tokens, forming a single prompt for the LLM.

### Task 1.3: Structural Token Analysis

Complete the cell below to inspect the input_ids tensor generated from the last processed sample. Your task is to use this tensor to calculate the exact number of visual tokens and text tokens present in the sequence.



In [ ]:
input_ids = inputs.input_ids

# TODO: Find the token ID for image pads
image_token_id = ...

# TODO: Count visual tokens and text tokens
num_visual_tokens = ...
num_text_tokens = ...

print(f"Visual Tokens: {num_visual_tokens} | Text Tokens: {num_text_tokens}")

**Question 1.3a (Empirical Analysis):** Review the printed token counts from the code execution above. Calculate the exact percentage of the total input sequence length ($N$) that is occupied by the visual tokens versus the textual tokens. What does this distribution tell you about how a VLM allocates its processing capacity compared to a text-only LLM?

**Student's Answer:**



**Question 1.3b (Mathematical Complexity Analysis):** Consider the standard computational cost formula per forward pass for a single Transformer processing an input sequence:

$$\text{Total FLOPs} = T \times (4nd^2 + 2n^2d + 2ndm)$$

Where:

- $n$ is the sequence length (total number of tokens).
- $d$ is the model's hidden embedding dimension size.
- $m$ is the intermediate dimension size of the MLP layer.
- $T$ is the number of transformer blocks/layers.

Based on this formula and the token counts you extracted in Task 1.3, identify which specific term inside the parentheses poses the greatest computational threat when shifting from a text-only LLM to a Vision-Language Model. Explain why processing visual data scales the computational overhead so aggressively.

**Student's Answer:**


# S2: Dynamic Resolution and Efficient Inference

**Objective:** In this section, you will explore how Qwen processes images, manage memory and accelerate inference speed by controlling image resolution.

## 1. Dynamic Resolution Mechanism

Older Vision-Language Models required all input images to be resized or cropped to a fixed resolution before processing. Qwen utilizes a more advanced approach called Dynamic Resolution.

### Task 2.1: Dynamic Resolution

Review the official documentation or technical reports for the Qwen-VL architecture (e.g., the [Qwen2-VL Paper](https://arxiv.org/abs/2409.12191) or [Qwen2.5-VL Blog Post](https://qwen.ai/blog?id=qwen2.5-vl)). Write a brief paragraph explaining how Qwen handles images of varying sizes and aspect ratios. How does it preserve the original image structure without forcing a rigid resize?

**Student's Answer:**



## 2. Baseline Inference (High Resolution)

To measure the impact of resolution on speed and accuracy, we first need to establish a baseline. We will evaluate the model on 100 samples using its default, unrestricted resolution settings.

### Task 2.2a: Prepare the Evaluation Subset

Complete and run the cell below to randomly select 100 image-based questions from the ScienceQA dataset. Crucially, we extract these samples from the test split to prevent any data leakage during the training phase later.


In [ ]:
test_image_data = load_dataset("derek-thomas/ScienceQA")["test"].filter(lambda x: x['image'] is not None)

# TODO: Randomly sample 100 items from test_image_data (seed=42)
...
eval_samples = ...

### Task 2.2b: Run Baseline Evaluation

Complete the code below to run the inference loop on the 100 selected samples. We will use the standard processor from the previous section. The code includes a timer to track exactly how long the entire batch takes.


In [ ]:
model.eval()
correct_predictions = 0
start_time = time.time()

for sample in eval_samples:
    ground_truth = str(sample['choices'][sample['answer']]).strip()
    choices_text = "\n".join([f"- {c}" for c in sample['choices']])
    full_prompt = f"{sample['question']}\nChoices:\n{choices_text}\nAnswer concisely with the correct choice."

    # TODO: Build messages and apply chat template (disable thinking mode)
    ...
    
    # TODO: Process vision info and get model inputs
    ...

    # TODO: Generate outputs and decode the text
    ...
    output_text = ...

    if ground_truth.lower() in output_text.lower():
        correct_predictions += 1

total_time = time.time() - start_time
accuracy = (correct_predictions / len(eval_samples)) * 100
print(f"Baseline (High-Res) -> Time: {total_time:.2f}s | Accuracy: {accuracy:.2f}%")

## 3. Low-Resolution Inference

Now, we will restrict the image resolution before passing it to the model. By setting a specific `max_pixels` limit, we force the processor to generate fewer visual patches.

### Task 2.3: Re-initialize Processor and Re-evaluate

Run the cell below. It creates a new processor with a strict pixel limit and evaluates the exact same 100 samples. Observe how the time and accuracy metrics change.

(Note: The minimum acceptable resolution for `max_pixels` in this exercise is 32 * 28 * 28.)


In [ ]:
# TODO: Initialize processor_low_res (max_pixels = 32 * 28 * 28)
processor_low_res = ...

# TODO: Re-write/copy the evaluation loop using processor_low_res
...

## 4. Mathematical Complexity Analysis

You should have noticed a difference in execution time and perhaps accuracy between the baseline and the low-resolution evaluation. Let's analyze exactly why this happens.

### Task 2.4: Comparative and Root Cause Analysis

First, create a simple chart or table (you can use Python or write it directly in your text answer) comparing the execution time and accuracy between the Baseline (High Resolution) and the Low-Resolution inference.

Based on your comparison:

1. **Accuracy Change:** How did the accuracy change? If there was a drop in accuracy, was it severe (causing the model to completely lose its ability to answer correctly), or was it minimal? What conclusion can you draw from this regarding the model's visual reasoning capabilities at lower resolutions?

2. **Time Change:** Recalling the Total FLOPs formula from the previous section, explain the exact reason for the change in execution time. Why does reducing the image resolution lead to an increase in inference speed?

**Student's Answer:**



---

**Important Notice for the Next Section**

In the upcoming section (Section 3), we plan to fine-tune the model on the ScienceQA dataset. Because training a model requires significantly more memory (RAM/VRAM) to store gradients compared to simple inference, you are highly likely to encounter Out of Memory (OOM) errors if you attempt to train on high-resolution images.

To bypass this hardware limitation, we will be forced to decrease the image resolution during the fine-tuning process (e.g., using the `max_pixels = 32 * 28 * 28` setting).

**Crucial Note:** For your final evaluation at the end of Section 3 to be scientifically valid, the restricted resolution you utilize here in Task 2.3 must be exactly the same as the resolution you will use during the fine-tuning phase. This ensures a strictly accurate comparison between the pre-trained and fine-tuned models.

# S3: Supervised Fine-Tuning (with LoRA)

**Objective:** In this section, you will implement Low-Rank Adaptation (LoRA) to fine-tune the Qwen3.5 0.8B model. You will format multimodal data for training, configure memory-efficient training parameters, and evaluate the final fine-tuned model against your previous baseline.

## 1. Preparing the Training Data

To train the model, we need to convert the raw dataset into the exact chat structure the model expects.

### Task 3.1: Data Formatting and Structure

Complete the code below to extract a 1,000-sample subset from the ScienceQA dataset for training. Crucially, ensure this subset strictly contains image-based questions, as our goal is to train the visual reasoning capabilities of the model.

In [ ]:
train_image_data = load_dataset("derek-thomas/ScienceQA")["train"].filter(lambda x: x['image'] is not None)
random.seed(42)
train_subset = random.sample(list(train_image_data), 1000)

formatted_train_data = []

for sample in train_subset:
    ground_truth = str(sample['choices'][sample['answer']]).strip()
    choices_text = "\n".join([f"- {c}" for c in sample['choices']])
    full_prompt = f"{sample['question']}\nChoices:\n{choices_text}\nAnswer concisely with the correct choice."
    
    # TODO: Build messages list ('user' gets image+prompt, 'assistant' gets ground_truth)
    messages = ...
    
    formatted_train_data.append({"messages": messages})

## 2. LoRA Configuration for VLMs

Fine-tuning an entire Vision-Language Model on standard hardware is extremely difficult due to massive memory requirements. We will use the `peft` library to apply Low-Rank Adaptation (LoRA), which freezes the base model and only trains a tiny set of injected weight matrices.

### Task 3.2: Configure PEFT

Review the Qwen3.5 architecture and standard LoRA practices. You need to identify which specific layers within the Large Language Model component should be targeted for adaptation. Do not target the Vision component layers.


In [ ]:
from peft import LoraConfig, get_peft_model

# TODO: Configure LoRA (target ONLY LLM layers)
peft_config = ...

# TODO: Apply PEFT model and print trainable parameters
...

## 3. The Training Loop

We will use Hugging Face's `Trainer` mapped for VLMs to handle the training loop. Because training memory footprint is heavily dependent on sequence length, we must strictly manage hardware constraints. To make the process clear, we will break the training setup into three smaller steps.

### Task 3.3a: Setup the Processor and Data Collator

Complete and run the cell below to load our restricted-resolution processor and define a custom `DataCollator` that processes multimodal inputs dynamically during training.


In [ ]:
# Load low-res processor for memory efficiency during training
processor_train = AutoProcessor.from_pretrained(model_id, max_pixels=32 * 28 * 28)

def multimodal_data_collator(features):
    texts, images = [], []
    for feature in features:
        # TODO: Apply chat template  and extract vision info
        ...
        
        texts.append(text)
        if image_inputs: images.extend(image_inputs)
        
    # TODO: Process batch using processor_train
    batch = ...
    
    # TODO: Set labels for Causal LM
    batch["labels"] = ...
    
    return batch

### Task 3.3b: Configure Training Arguments

The acceptable training configurations are specified within the code comments below. If your hardware encounters memory limitations (such as Out of Memory errors) or exceptionally slow training times, you should lower these parameters accordingly to successfully complete the fine-tuning process. Run the cell below to configure them.




In [ ]:
# TODO: Configure TrainingArguments (handle batch_size and gradient_accumulation carefully for OOM)
training_args = ...

### Task 3.3c: Initialize Trainer and Execute

Run the cell below to piece everything together into the `Trainer` and start the Fine-Tuning process.


In [ ]:
# TODO: Initialize Trainer with model, args, dataset, and collator
trainer = ...

# TODO: Start training
...

# TODO: Save the fine-tuned model
...

## 4. Secondary Evaluation

With the model fine-tuned, it is time to measure the impact of our training. We will evaluate the fine-tuned model using the exact same evaluation dataset and settings from Section 2.

### Task 3.4: Evaluate the Fine-Tuned Model

Run the exact same 100 evaluation samples from Task 2.2 using the newly fine-tuned model and the restricted low-resolution processor.



In [ ]:
# TODO: Evaluate the fine-tuned model on eval_samples using processor_train
...

# S4: Critical Analysis
---

### Task 4.1: The Tripartite Trade-off

Throughout this exercise, you have gathered metrics for three distinct states of the model. Fill in the comparison table below with the data you collected:

| Model State | Resolution (max_pixels) | Total Inference Time (s) | Accuracy (%) |
| --- | --- | --- | --- |
| **A) Baseline (High Res)** | Default (Unrestricted) |  |   |
| **B) Baseline (Low Res)** | 32 * 28 * 28 |  |  |
| **C) Fine-Tuned (Low Res)** | 32 * 28 * 28 |  |  |

Based on the data in your table, analyze the "Speed-Accuracy Trade-off." Did the computational time and effort spent fine-tuning the model in Section 3 justify the final inference speed and accuracy achieved in state (C)?

**Student's Answer:**


---

### Task 4.2: Understanding Visual Redundancy
Look at your table from Task 4.1 and compare the rate of change in resolution versus the rate of change in accuracy when moving from State (A) to State (B). 

In machine learning, this specific behavior is related to the concept of data "redundancy." Based on your results, explain what visual redundancy means. Why can a model still logically process an image after losing a massive percentage of its visual patches, whereas dropping the exact same percentage of words from a text prompt would completely destroy its meaning?

**Student's Answer:**



### Task 4.3: Exploring the Frontiers of Efficiency

In this exercise, we relied heavily on "Dynamic Resolution Reduction" to fit the VLM inference and training onto constrained hardware. However, this is just one approach in the broader field of efficient AI.

Research the current literature on Large Vision-Language Models. Identify and name at least **two other techniques** used to accelerate inference and reduce memory (VRAM) consumption. Provide a concise, one-sentence explanation of the core mechanism for each technique.

**Student's Answer:**


### Task 4.4: Task-Specific Resolution Sensitivity
In our experiment, you saw how reducing the resolution affected the accuracy on the ScienceQA dataset. Do you think this result would be the same for all types of visual tasks? 

Compare the ScienceQA dataset to a dataset focused on reading dense charts, complex graphs, or document text (e.g., ChartQA or DocVQA). How would extreme resolution reduction affect the accuracy in those cases, and why? 

*(Note: You do not need to write code or run any tests for this question. Just provide your analytical opinion).*

**Student's Answer:**

